# Ejercicio: Asistente de Documentos con Gemini y Gradio
 
**Tiempo:** 40-45 minutos  
**Grupos:** 3-4 personas  
 
## Contexto
 
En los notebooks anteriores construimos chatbots que responden preguntas generales.
En este ejercicio daremos un paso más: construir un asistente que responda preguntas
sobre un documento específico — en este caso, el paper seminal de los Transformers:
**"Attention is All You Need"** (Vaswani et al., 2017).
 
Esta técnica — inyectar el contenido de un documento en el contexto del modelo —
es la base conceptual de **RAG (Retrieval-Augmented Generation)**, uno de los
patrones más usados en aplicaciones de IA en producción.
 
## Objetivo
 
Construir una aplicación Gradio donde el usuario pueda:
1. Cargar el PDF del paper
2. Hacer preguntas sobre su contenido
3. Obtener respuestas basadas **exclusivamente** en el documento
 
## Lo que van a aprender
 
- Extracción de texto desde PDFs con `pypdf`
- Inyección de contexto externo en el system prompt
- Limitaciones de este enfoque y por qué existe RAG
- Integración de `gr.File` en una interfaz Gradio
 
---

## Paso 0: Instalación y configuración
 
### 0.1 Instala las dependencias necesarias
 
Necesitarás tres bibliotecas nuevas además de las que ya conoces:
- `pypdf`: para extraer texto de archivos PDF
- `gradio`: para la interfaz web
- `google-genai`: para el modelo
- `sentence-transformers`: para embeddings locales (Versión 4 RAG)
 
```
pip install pypdf gradio google-genai python-dotenv sentence-transformers
```

In [1]:
# %%bash
# pip install pypdf gradio google-genai python-dotenv sentence-transformers


### 0.2 Descarga el paper
 
Descarga el PDF de ArXiv (acceso abierto):
```
https://arxiv.org/pdf/1706.03762
```
 
Guárdalo en la misma carpeta que este notebook con el nombre `attention_is_all_you_need.pdf`.
 
### 0.3 Configura tus credenciales
 
Crea un archivo `.env` con tu API key de Gemini:
```
GEMINI_API_KEY="tu_key_aqui"
```

## Paso 1: Extracción de texto del PDF

Lo primero es leer el PDF y extraer su contenido como texto plano.
`pypdf` hace esto en pocas líneas.

**Instrucciones:**
1. Importa `PdfReader` desde `pypdf`
2. Crea una función `extract_text_from_pdf(pdf_path)` que:
   - Abra el PDF desde la ruta `pdf_path`
   - Itere sobre todas las páginas
   - Concatene el texto de cada página
   - Retorne el texto completo como string
3. Prueba la función con `attention_is_all_you_need.pdf`
4. Imprime los primeros 500 caracteres para verificar que funcionó

**Pista:** `PdfReader` tiene un atributo `pages` que es una lista.
Cada página tiene un método `extract_text()`.

In [2]:
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path: str) -> str:
    """Extrae todo el texto de un archivo PDF.
    
    Args:
        pdf_path: Ruta al archivo PDF
    
    Returns:
        Texto completo del PDF como string, todas las páginas concatenadas
    """
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        # extract_text() retorna None en páginas solo-imagen; "or ''" evita TypeError
        text += (page.extract_text() or "") + "\n"
    return text

# Prueba la función
pdf_path = "data/attention_is_all_you_need.pdf"
document_text = extract_text_from_pdf(pdf_path)

# Estadísticas
reader = PdfReader(pdf_path)
print(f"Numero de paginas: {len(reader.pages)}")
print(f"Caracteres extraidos: {len(document_text):,}")

# Verificar primeros 500 caracteres
print("\n--- Primeros 500 caracteres ---")
print(document_text[:500])

Numero de paginas: 15
Caracteres extraidos: 39,630

--- Primeros 500 caracteres ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 


## Paso 2: Inicialización del cliente de Gemini

Igual que en los notebooks anteriores.

**Instrucciones:**
1. Importa las bibliotecas necesarias (`os`, `dotenv`, `google.genai`, `google.genai.types`)
2. Carga las variables de entorno
3. Inicializa el cliente de Gemini
4. Define la constante `MODELO = "gemini-2.5-flash-lite"`
5. Verifica que la API key esté disponible

In [3]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Cargar variables de entorno desde .env
load_dotenv()

# Verificar que la API key esté disponible
if os.getenv("GEMINI_API_KEY"):
    print("Gemini API Key cargada correctamente")
else:
    print("ERROR: GEMINI_API_KEY no encontrada. Verifica tu archivo .env")

# Inicializar el cliente de Gemini
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# Modelo que usaremos en todo el notebook
MODELO = "gemini-2.5-flash-lite"

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Gemini API Key cargada correctamente


## Paso 3: Función de chat con contexto del documento

Esta es la parte central del ejercicio. La idea es construir un system prompt
que incluya el texto completo del paper, instruyendo al modelo a responder
**solo** basándose en ese contenido.

**Instrucciones:**
1. Crea una función `build_system_prompt(document_text)` que reciba el texto
   del documento y retorne un system prompt que:
   - Defina el rol del asistente (experto en el paper)
   - Incluya el texto completo del documento
   - Instruya al modelo a responder **solo** con información del documento
   - Indique qué hacer si la respuesta no está en el documento

2. Crea una función `chat_con_documento(message, history, document_text)` que:
   - Construya el historial en formato Gemini (objetos `types.Content`)
   - Use `generate_content_stream` con el system prompt del documento
   - Retorne la respuesta con `yield` para streaming

**Pista:** Recuerda que en Gradio el historial llega como lista de dicts
con claves `role` y `content`. El rol del asistente en Gemini es `"model"`.

In [4]:
def get_text(content) -> str:
    """Extrae texto de un contenido de mensaje de Gradio,
    sin importar si llega como string o lista de dicts."""
    if isinstance(content, str):
        return content
    elif isinstance(content, list):
        return content[0].get("text", "") if content else ""
    return str(content)


def build_system_prompt(document_text: str) -> str:
    """Construye el system prompt que incluye el texto completo del documento."""
    return f"""Eres un experto asistente académico especializado en el paper
"Attention is All You Need" (Vaswani et al., 2017).

A continuación se te proporciona el texto COMPLETO del paper:

--- INICIO DEL DOCUMENTO ---
{document_text}
--- FIN DEL DOCUMENTO ---

INSTRUCCIONES IMPORTANTES:
1. Responde EXCLUSIVAMENTE con información que esté en el documento proporcionado arriba.
2. Si la pregunta del usuario no puede ser respondida con la información del documento,
   responde: "La información solicitada no se encuentra en el documento proporcionado."
3. NO uses tu conocimiento general para responder, incluso si conoces la respuesta.
4. Cuando sea relevante, menciona la sección o parte del paper donde se encuentra la información.
5. Responde en el mismo idioma en que el usuario hace la pregunta.
6. Incluye siempre una cita textual del documento (entre comillas) que respalde tu respuesta."""


def chat_con_documento(message: str, history: list, document_text: str):
    """Función de chat que Gradio llama cada vez que el usuario envía un mensaje.
    
    Args:
        message: Mensaje actual del usuario
        history: Historial en formato Gradio 5.x
        document_text: Texto del documento (pasado vía additional_inputs)
    
    Yields:
        Respuesta acumulada del modelo (streaming)
    """
    # 1. Construir system prompt con el documento completo incrustado
    system_prompt = build_system_prompt(document_text)

    # 2. Convertir historial de Gradio al formato Gemini (lista de types.Content)
    contenido = []
    for entry in history:
        if isinstance(entry, dict):
            # MAPEO CRÍTICO: Gradio usa "assistant", Gemini requiere "model"
            # Si se envía "assistant", la API lanza error de rol inválido
            role = "model" if entry["role"] == "assistant" else "user"
            contenido.append(
                types.Content(
                    role=role,
                    parts=[types.Part(text=get_text(entry["content"]))]
                )
            )
        elif isinstance(entry, (list, tuple)) and len(entry) == 2:
            # Compatibilidad con formato alternativo [user_msg, assistant_msg]
            user_msg, assistant_msg = entry
            contenido.append(
                types.Content(role="user", parts=[types.Part(text=get_text(user_msg))])
            )
            if assistant_msg:
                contenido.append(
                    types.Content(role="model", parts=[types.Part(text=get_text(assistant_msg))])
                )

    # 3. Agregar mensaje actual del usuario al final del historial
    contenido.append(
        types.Content(role="user", parts=[types.Part(text=message)])
    )

    # 4. Llamar a Gemini en modo streaming (yield activa streaming en Gradio automáticamente)
    respuesta = ""
    for chunk in client.models.generate_content_stream(
        model=MODELO,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,  # Directiva de comportamiento, más peso que el user prompt
        ),
        contents=contenido
    ):
        if chunk.text:
            respuesta += chunk.text
            yield respuesta  # Gradio detecta yield → muestra tokens progresivamente

## Paso 4: Interfaz Gradio

Ahora construimos la interfaz. Usaremos `gr.ChatInterface` con `additional_inputs`
para pasar el texto del documento extraído.

**Instrucciones:**
1. Extrae el texto del PDF usando la función del Paso 1
2. Imprime cuántas páginas tiene y cuántos caracteres extraíste
3. Construye la interfaz con `gr.ChatInterface`:
   - `fn`: tu función `chat_con_documento`
   - `title`: un título descriptivo
   - `description`: explica qué puede hacer el asistente
   - `additional_inputs`: un `gr.Textbox` visible=False que contenga el texto
     del documento (así Gradio lo pasa automáticamente a la función)
   - `examples`: al menos 3 preguntas relevantes sobre el paper

**Pista:** Para pasar el texto del documento sin mostrarlo en la interfaz:
```python
gr.Textbox(value=document_text, visible=False)
```

4. Lanza la interfaz con `demo.launch(server_name="0.0.0.0", server_port=8080)`

In [5]:
import gradio as gr

# Extraer texto del PDF
pdf_path = "data/attention_is_all_you_need.pdf"
document_text = extract_text_from_pdf(pdf_path)

reader = PdfReader(pdf_path)
print(f"Documento cargado: {len(reader.pages)} paginas, {len(document_text):,} caracteres")

# Conteo real de tokens con la API de Gemini (Mejora A integrada en la interfaz)
LIMITE = 1_000_000  # Gemini 2.5 Flash: 1M tokens de contexto
tokens_doc = client.models.count_tokens(
    model=MODELO, contents=build_system_prompt(document_text)
).total_tokens
print(f"Tokens del sistema: {tokens_doc:,} / {LIMITE:,} ({tokens_doc / LIMITE * 100:.2f}%)")

with gr.Blocks(title="Asistente: Attention is All You Need") as demo:
    gr.Markdown("# Asistente del Paper: Attention is All You Need")
    gr.Markdown(
        'Pregunta cualquier cosa sobre el paper "Attention is All You Need" '
        "(Vaswani et al., 2017). El asistente responde **exclusivamente** con información del documento."
    )
    gr.Markdown(
        f"**Tokens del sistema:** `{tokens_doc:,}` / `{LIMITE:,}` "
        f"({tokens_doc / LIMITE * 100:.2f}% del límite) — "
        f"`{LIMITE - tokens_doc:,}` tokens disponibles para conversación"
    )
    gr.ChatInterface(
        fn=chat_con_documento,
        additional_inputs=[
            gr.Textbox(value=document_text, visible=False)
        ],
        examples=[
            ["¿Cuál es la arquitectura principal propuesta en el paper?"],
            ["¿Qué es el mecanismo de atención multi-cabeza?"],
            ["¿Cuántas capas tiene el encoder del modelo base?"],
            ["¿Quiénes son los autores del paper?"],
            ["¿Cuál es el resultado del modelo en WMT 2014 English-to-German?"],
        ],
        flagging_mode="never"
    )

demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)

Documento cargado: 15 paginas, 39,630 caracteres
Tokens del sistema: 11,040 / 1,000,000 (1.10%)
* Running on local URL:  http://0.0.0.0:8080
* To create a public link, set `share=True` in `launch()`.


## Paso 5: Prueba y reflexión
 
Una vez que la interfaz esté funcionando, prueba estas preguntas:
 
1. *"¿Cuál es la arquitectura principal propuesta en el paper?"*
2. *"¿Qué es el mecanismo de atención?"*
3. *"¿Cuántas capas tiene el encoder del modelo base?"*
4. *"¿Quiénes son los autores del paper?"*
5. *"¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?"*
 
Y esta pregunta trampa:
6. *"¿Qué es GPT-4?"*
 
Esta última pregunta **no está en el paper**. Observa cómo responde el modelo.
¿Usa su conocimiento general o respeta la instrucción de ceñirse al documento?

## Paso 6 (Adicional): Mejora el sistema

Las cuatro mejoras opcionales fueron implementadas:

**A) Indicador de tokens** ✅ **(Implementada)**
Usa `client.models.count_tokens()` para obtener el conteo real de tokens del system prompt
desde la API de Gemini (no una estimación). Muestra cuántos tokens usa el paper y qué
porcentaje representan del límite de 1,000,000 tokens de Gemini 2.5 Flash.

**B) Subida dinámica de PDF** ✅ **(Implementada)**
Nueva interfaz en el puerto 8081 con `gr.File` en `additional_inputs`. Cuando el usuario
sube un PDF, `chat_con_documento_v2` extrae su texto y lo usa como documento base.
Si no se sube nada, usa el paper "Attention is All You Need" por defecto.

**C) Citas del documento** ✅ **(Implementada)**
Instrucción 6 del system prompt obliga al modelo a incluir una cita textual del paper
entre comillas en cada respuesta, haciendo las respuestas verificables.

**D) Multi-PDF con citación de fuente** ✅ **(Implementada)**
Nueva interfaz en el puerto 8082 que carga todos los PDFs de la carpeta `data/` y permite
subir PDFs adicionales. Cuando responde, el modelo indica de cuál archivo proviene cada
afirmación usando el formato `[nombre_archivo.pdf]`, permitiendo consultar múltiples
documentos simultáneamente.

In [6]:
# ============================================================
# MEJORA A: Indicador de tokens (conteo real con la API de Gemini)
# ============================================================

system_prompt_completo = build_system_prompt(document_text)

# count_tokens llama a la API y retorna el conteo exacto (no estimación)
token_response = client.models.count_tokens(
    model=MODELO,
    contents=system_prompt_completo
)
tokens_reales = token_response.total_tokens
LIMITE = 1_000_000  # Gemini 2.5 Flash: 1M tokens de contexto

print(f"=== INDICADOR DE TOKENS — MEJORA A ===")
print(f"Tokens del system prompt (conteo Gemini): {tokens_reales:,}")
print(f"Límite del modelo ({MODELO}):              {LIMITE:,}")
print(f"Porcentaje usado:                          {tokens_reales / LIMITE * 100:.2f}%")
print(f"Tokens disponibles para conversación:      {LIMITE - tokens_reales:,}")
print()
print(f"Conclusión: el paper usa solo el {tokens_reales/LIMITE*100:.1f}% del límite.")
print(f"Quedan {LIMITE - tokens_reales:,} tokens libres para el historial de chat.")

=== INDICADOR DE TOKENS — MEJORA A ===
Tokens del system prompt (conteo Gemini): 11,040
Límite del modelo (gemini-2.5-flash-lite):              1,000,000
Porcentaje usado:                          1.10%
Tokens disponibles para conversación:      988,960

Conclusión: el paper usa solo el 1.1% del límite.
Quedan 988,960 tokens libres para el historial de chat.


In [7]:
# ============================================================
# MEJORA B: Subida dinámica de PDF con indicador de tokens (Mejora A integrada)
# ============================================================

def chat_con_documento_v2(message: str, history: list, pdf_file, document_text_default: str):
    """Versión mejorada: acepta subida dinámica de PDF vía gr.File."""
    texto = extract_text_from_pdf(pdf_file) if pdf_file is not None else document_text_default
    system_prompt = build_system_prompt(texto)

    contenido = []
    for entry in history:
        if isinstance(entry, dict):
            role = "model" if entry["role"] == "assistant" else "user"
            contenido.append(
                types.Content(role=role, parts=[types.Part(text=get_text(entry["content"]))])
            )
        elif isinstance(entry, (list, tuple)) and len(entry) == 2:
            u, a = entry
            contenido.append(types.Content(role="user", parts=[types.Part(text=get_text(u))]))
            if a:
                contenido.append(types.Content(role="model", parts=[types.Part(text=get_text(a))]))

    contenido.append(types.Content(role="user", parts=[types.Part(text=message)]))

    respuesta = ""
    for chunk in client.models.generate_content_stream(
        model=MODELO,
        config=types.GenerateContentConfig(system_instruction=system_prompt),
        contents=contenido
    ):
        if chunk.text:
            respuesta += chunk.text
            yield respuesta


def actualizar_tokens_v2(pdf_file) -> str:
    """Calcula el conteo real de tokens para el documento activo y retorna Markdown."""
    if pdf_file is not None:
        texto = extract_text_from_pdf(pdf_file)
        nombre = os.path.basename(pdf_file)
    else:
        texto = document_text
        nombre = "attention_is_all_you_need.pdf (base)"
    tokens = client.models.count_tokens(
        model=MODELO, contents=build_system_prompt(texto)
    ).total_tokens
    pct = tokens / LIMITE * 100
    return (
        f"**Tokens del sistema (`{nombre}`):** `{tokens:,}` / `{LIMITE:,}` "
        f"({pct:.2f}%) — `{LIMITE - tokens:,}` disponibles para conversación"
    )


# Conteo inicial para el documento base
tok_base = client.models.count_tokens(
    model=MODELO, contents=build_system_prompt(document_text)
).total_tokens

with gr.Blocks(title="Asistente — Subida Dinámica de PDF") as demo_v2:
    gr.Markdown("## Asistente de Documentos — Subida Dinámica de PDF")
    gr.Markdown(
        "Sube cualquier PDF y haz preguntas sobre su contenido. "
        "Si no subes ningún archivo, el asistente responde sobre 'Attention is All You Need'."
    )
    token_display = gr.Markdown(
        f"**Tokens del sistema (doc. base):** `{tok_base:,}` / `{LIMITE:,}` "
        f"({tok_base / LIMITE * 100:.2f}%) — `{LIMITE - tok_base:,}` disponibles"
    )
    chat_v2 = gr.ChatInterface(
        fn=chat_con_documento_v2,
        additional_inputs=[
            gr.File(label="Sube un PDF (opcional)", file_types=[".pdf"]),
            gr.Textbox(value=document_text, visible=False),
        ],
        examples=[
            ["¿De qué trata este documento?", None, None],
            ["Haz un resumen de las ideas principales", None, None],
            ["¿Cuáles son las conclusiones del documento?", None, None],
        ],
        flagging_mode="never"
    )
    # Actualizar el indicador cuando el usuario sube o quita un PDF
    chat_v2.additional_inputs[0].change(
        fn=actualizar_tokens_v2,
        inputs=[chat_v2.additional_inputs[0]],
        outputs=[token_display]
    )

demo_v2.launch(server_name="0.0.0.0", server_port=8081, show_error=True)

* Running on local URL:  http://0.0.0.0:8081
* To create a public link, set `share=True` in `launch()`.


In [8]:
# ============================================================
# MEJORA D: Multi-PDF — responde de cualquier documento y cita la fuente
# ============================================================

def load_pdfs_from_folder(folder: str) -> dict:
    """Carga todos los PDFs de una carpeta. Retorna {filename: text}."""
    docs = {}
    if os.path.isdir(folder):
        for fname in sorted(os.listdir(folder)):
            if fname.lower().endswith(".pdf"):
                fpath = os.path.join(folder, fname)
                try:
                    docs[fname] = extract_text_from_pdf(fpath)
                except Exception as e:
                    print(f"Error leyendo {fname}: {e}")
    return docs


def build_system_prompt_multi(docs: dict) -> str:
    """Construye system prompt para múltiples documentos con etiquetas de origen."""
    bloques = []
    for i, (nombre, texto) in enumerate(docs.items(), 1):
        bloques.append(
            f"=== DOCUMENTO {i}: {nombre} ===\n{texto}\n=== FIN DOCUMENTO {i}: {nombre} ==="
        )
    nombres = ", ".join(docs.keys())
    return f"""Eres un asistente experto en análisis de documentos académicos.
Tienes acceso a {len(docs)} documento(s): {nombres}

{chr(10).join(bloques)}

INSTRUCCIONES IMPORTANTES:
1. Responde EXCLUSIVAMENTE con información que esté en los documentos proporcionados.
2. SIEMPRE indica de cuál documento proviene cada afirmación con el formato [nombre_archivo.pdf].
   Ejemplo: "Según [attention_is_all_you_need.pdf], el modelo usa 6 capas de encoder."
3. Si la respuesta involucra varios documentos, cita cada uno con su etiqueta.
4. Si la información no está en ningún documento, responde exactamente:
   "La información no se encuentra en los documentos cargados."
5. NO uses conocimiento general; cíñete al contenido de los documentos.
6. Incluye citas textuales (entre comillas) junto a la etiqueta del documento fuente."""


def chat_multi_pdf(message: str, history: list, uploaded_files, use_defaults: bool):
    """Chat que responde de múltiples PDFs e indica la fuente de cada respuesta."""
    docs = {}
    if use_defaults:
        docs.update(base_docs)
    if uploaded_files:
        files = uploaded_files if isinstance(uploaded_files, list) else [uploaded_files]
        for f in files:
            nombre = os.path.basename(f)
            try:
                docs[nombre] = extract_text_from_pdf(f)
            except Exception as e:
                docs[nombre] = f"[Error al leer el archivo: {e}]"
    if not docs:
        yield "No hay documentos cargados. Sube al menos un PDF o activa los documentos base."
        return

    system_prompt = build_system_prompt_multi(docs)
    contenido = []
    for entry in history:
        if isinstance(entry, dict):
            role = "model" if entry["role"] == "assistant" else "user"
            contenido.append(types.Content(role=role, parts=[types.Part(text=get_text(entry["content"]))]))
        elif isinstance(entry, (list, tuple)) and len(entry) == 2:
            u, a = entry
            contenido.append(types.Content(role="user", parts=[types.Part(text=get_text(u))]))
            if a:
                contenido.append(types.Content(role="model", parts=[types.Part(text=get_text(a))]))
    contenido.append(types.Content(role="user", parts=[types.Part(text=message)]))

    respuesta = ""
    for chunk in client.models.generate_content_stream(
        model=MODELO,
        config=types.GenerateContentConfig(system_instruction=system_prompt),
        contents=contenido
    ):
        if chunk.text:
            respuesta += chunk.text
            yield respuesta


def actualizar_tokens_multi(uploaded_files, use_defaults: bool) -> str:
    """Calcula tokens del sistema para la combinación actual de documentos."""
    docs = {}
    if use_defaults:
        docs.update(base_docs)
    if uploaded_files:
        files = uploaded_files if isinstance(uploaded_files, list) else [uploaded_files]
        for f in files:
            nombre = os.path.basename(f)
            try:
                docs[nombre] = extract_text_from_pdf(f)
            except Exception:
                pass
    if not docs:
        return "**Sin documentos cargados.** Sube al menos un PDF o activa los documentos base."
    tokens = client.models.count_tokens(
        model=MODELO, contents=build_system_prompt_multi(docs)
    ).total_tokens
    pct = tokens / LIMITE * 100
    nombres = ", ".join(docs.keys())
    return (
        f"**Tokens del sistema ({nombres}):** `{tokens:,}` / `{LIMITE:,}` "
        f"({pct:.2f}%) — `{LIMITE - tokens:,}` disponibles"
    )


# Pre-cargar todos los PDFs de la carpeta data/
base_docs = load_pdfs_from_folder("data")
print(f"Documentos base cargados: {len(base_docs)}")
for nombre, texto in base_docs.items():
    print(f"  - {nombre} ({len(texto):,} caracteres)")

# Conteo inicial para todos los documentos base
if base_docs:
    tok_multi_base = client.models.count_tokens(
        model=MODELO, contents=build_system_prompt_multi(base_docs)
    ).total_tokens
    tok_multi_str = (
        f"**Tokens del sistema ({', '.join(base_docs.keys())}):** "
        f"`{tok_multi_base:,}` / `{LIMITE:,}` "
        f"({tok_multi_base / LIMITE * 100:.2f}%) — `{LIMITE - tok_multi_base:,}` disponibles"
    )
else:
    tok_multi_str = "**Sin documentos base.** Sube al menos un PDF."

with gr.Blocks(title="Asistente Multi-Documento") as demo_multi:
    gr.Markdown("## Asistente Multi-Documento — Cita la fuente automáticamente")
    gr.Markdown(
        "Pregunta sobre cualquier documento cargado. "
        "El asistente indica de cuál PDF proviene cada respuesta usando `[nombre.pdf]`.\n\n"
        f"Documentos base: {', '.join(base_docs.keys()) if base_docs else 'ninguno'}"
    )
    token_display_multi = gr.Markdown(tok_multi_str)
    chat_multi = gr.ChatInterface(
        fn=chat_multi_pdf,
        additional_inputs=[
            gr.File(
                label="Agregar PDFs adicionales (opcional)",
                file_types=[".pdf"],
                file_count="multiple",
            ),
            gr.Checkbox(label="Incluir documentos base de la carpeta data/", value=True),
        ],
        examples=[
            ["¿De qué trata cada documento?", None, True],
            ["¿Cuál es la idea principal del paper sobre transformers?", None, True],
            ["Compara los enfoques de los diferentes documentos", None, True],
        ],
        flagging_mode="never",
    )
    # Actualizar el indicador cuando cambian los PDFs subidos o el checkbox
    chat_multi.additional_inputs[0].change(
        fn=actualizar_tokens_multi,
        inputs=[chat_multi.additional_inputs[0], chat_multi.additional_inputs[1]],
        outputs=[token_display_multi]
    )
    chat_multi.additional_inputs[1].change(
        fn=actualizar_tokens_multi,
        inputs=[chat_multi.additional_inputs[0], chat_multi.additional_inputs[1]],
        outputs=[token_display_multi]
    )

demo_multi.launch(server_name="0.0.0.0", server_port=8082, show_error=True)

Documentos base cargados: 1
  - attention_is_all_you_need.pdf (39,630 caracteres)
* Running on local URL:  http://0.0.0.0:8082
* To create a public link, set `share=True` in `launch()`.


## Mejora D: Multi-PDF con citación automática de fuente

Extiende el sistema para manejar **múltiples documentos simultáneamente**.
Cuando el usuario hace una pregunta, el asistente responde citando explícitamente
de cuál PDF proviene cada afirmación usando el formato `[nombre_archivo.pdf]`.

- Carga automática de todos los PDFs en la carpeta `data/`
- Subida dinámica de PDFs adicionales vía `gr.File(file_count="multiple")`
- Checkbox para activar/desactivar los documentos base
- Sistema prompt con bloques etiquetados por nombre de archivo

## Reflexión final

### 1. ¿Cuál es la limitación principal de este enfoque?

El enfoque actual inyecta **todo el texto del documento en el system prompt**. Esto funciona
mientras el documento quepa en la ventana de contexto del modelo. Para un paper de ~15 páginas
usamos ~13,000 tokens — solo el 1.3% del límite de 1,000,000 tokens de Gemini 2.5 Flash.

Sin embargo, con un documento de **1,000 páginas** (~866,000 tokens), estaríamos al 86.6% del
límite y pagaríamos esos tokens **en cada pregunta**. Con 2,000+ páginas superaríamos el límite
y el sistema fallaría completamente. Además, el costo de la API crece linealmente con el tamaño
del contexto enviado.

### 2. ¿Por qué existe RAG?

**RAG (Retrieval-Augmented Generation)** resuelve el problema de la limitación del contexto:

1. **Indexación previa**: El documento se divide en fragmentos pequeños ("chunks") y se convierten
   en vectores numéricos (*embeddings*) que capturan el significado semántico.
2. **Búsqueda por relevancia**: Cuando el usuario hace una pregunta, se buscan los chunks más
   similares semánticamente a esa pregunta usando distancia coseno entre vectores.
3. **Contexto selectivo**: Solo los chunks relevantes (~3-5 fragmentos) se inyectan en el prompt,
   no el documento completo.

Resultado: en vez de enviar 866,000 tokens por pregunta, se envían ~2,000 tokens con solo la
información pertinente. Escala a documentos de millones de páginas.

### 3. ¿Qué información podría "filtrarse" aunque el system prompt diga que no?

Gemini fue entrenado con texto de internet, que incluye el paper "Attention is All You Need"
(es un paper famoso y público desde 2017). Por tanto, el modelo **ya conoce** el contenido
del paper antes de leer el system prompt.

Si el system prompt dice "responde solo con el documento", el modelo tiende a obedecer, pero:
- Puede completar información que no está literalmente en el PDF (por ej., contexto histórico)
- No podemos distinguir si cita el documento o su conocimiento previo
- La instrucción 3 mitiga esto pero no garantiza aislamiento perfecto

**Para verificarlo:** Probar el mismo chatbot con un documento ficticio inventado (datos que el
modelo no puede conocer de su entrenamiento). Si responde correctamente a preguntas sobre ese
documento, está leyendo el contexto. Si "inventa" respuestas, está usando conocimiento previo.

---

## Recursos

- [Documentación de pypdf](https://pypdf.readthedocs.io)
- [Documentación de Gradio](https://www.gradio.app/docs)
- [Documentación de Gemini API](https://ai.google.dev/gemini-api/docs)
- [Paper original en ArXiv](https://arxiv.org/abs/1706.03762)


## Versión 4: Asistente RAG con embeddings reales

Esta versión implementa **RAG (Retrieval-Augmented Generation)** completo — el patrón
más usado en producción para sistemas de preguntas y respuestas sobre documentos.

### ¿Por qué RAG en vez de inyectar todo el documento?

El enfoque de las versiones anteriores (inyectar el documento completo en el prompt) funciona
bien para documentos pequeños, pero tiene limitaciones claras:

| Problema | Versiones 1–3 | Versión RAG |
|----------|--------------|-------------|
| Documentos grandes (> contexto) | Falla | Escala sin límite |
| Costo por pregunta | Proporcional al doc completo | Solo los fragmentos relevantes |
| Precisión | Modelo procesa todo el texto | Enfocado en lo relevante |
| Velocidad | Contexto largo → más lento | Contexto corto → más rápido |

### Cómo funciona el pipeline RAG

```
Documentos PDF
     │
     ▼
┌─────────────┐     chunk_size=800 chars
│  Chunking   │     overlap=150 chars (evita perder contexto en bordes)
└─────────────┘
     │  Lista de N fragmentos de texto
     ▼
┌─────────────┐     all-MiniLM-L6-v2 → vectores de 384 dimensiones
│  Embedding  │     Cada fragmento → 1 vector numérico que captura significado
└─────────────┘
     │  Matriz de embeddings (N × 384)
     ▼
┌─────────────┐     Almacenado en memoria como base_embeddings_rag
│   Índice    │
└─────────────┘

--- En tiempo de query ---

Pregunta del usuario
     │
     ▼
┌─────────────┐     Mismo modelo de embedding → vector de 384 dimensiones
│  Query emb  │
└─────────────┘
     │
     ▼
┌─────────────┐     Similitud coseno entre query y cada fragmento
│  Retrieval  │     Top-k fragmentos más relevantes (k=4)
└─────────────┘
     │  Solo los fragmentos relevantes (~2,000 tokens vs. ~11,000 del doc completo)
     ▼
┌─────────────┐     Prompt = sistema + fragmentos + pregunta
│  Gemini     │     Responde citando [FRAGMENTO N — archivo.pdf]
└─────────────┘
```

### Mejoras integradas en esta versión

| Mejora | Descripción |
|--------|-------------|
| **A** | Conteo real de tokens del contexto recuperado (no estimación) |
| **B** | Sube cualquier PDF y reconstruye el índice automáticamente |
| **C** | Cada respuesta cita el fragmento fuente con `[FRAGMENTO N — archivo.pdf]` |
| **D** | Combina múltiples documentos; indica de cuál proviene cada afirmación |


In [9]:
# Instala sentence-transformers para embeddings locales (sin API key)
# Solo necesario la primera vez; omite esta celda si ya está instalado
import subprocess
subprocess.run(["pip", "install", "sentence-transformers"], check=True)


CompletedProcess(args=['pip', 'install', 'sentence-transformers'], returncode=0)

In [10]:

# ============================================================
# VERSIÓN 4 — Infraestructura RAG: chunking + embeddings + índice
# ============================================================

import numpy as np
from sentence_transformers import SentenceTransformer

_st_model = SentenceTransformer("all-MiniLM-L6-v2")  # 80 MB, descarga una sola vez

def chunking(text: str, chunk_size: int = 800, overlap: int = 150) -> list:
      chunks, start = [], 0
      while start < len(text):
          end = min(start + chunk_size, len(text))
          chunks.append(text[start:end])
          if end == len(text):
              break
          start += chunk_size - overlap
      return chunks

def embed_texts(texts: list) -> np.ndarray:
      return _st_model.encode(texts, convert_to_numpy=True)

def build_index(docs: dict) -> tuple:
      all_chunks, all_meta = [], []
      for filename, text in docs.items():
          for i, chunk in enumerate(chunking(text)):
              all_chunks.append(chunk)
              all_meta.append({"file": filename, "idx": i})
      return all_chunks, embed_texts(all_chunks), all_meta

def buscar_chunks(query: str, chunks: list, embeddings, meta: list, k: int = 4) -> list:
      if not chunks or embeddings is None or len(embeddings) == 0:
          return []
      query_emb = embed_texts([query])[0]
      norms = np.linalg.norm(embeddings, axis=1) * np.linalg.norm(query_emb)
      norms = np.where(norms == 0, 1e-10, norms)
      similitudes = embeddings @ query_emb / norms
      top_k = np.argsort(similitudes)[::-1][:k]
      return [(chunks[i], meta[i], float(similitudes[i])) for i in top_k]

print("Construyendo índice RAG para los documentos base...")
base_chunks_rag, base_embeddings_rag, base_meta_rag = build_index(base_docs)
print(f"Índice RAG listo: {len(base_chunks_rag)} fragmentos")
for doc in base_docs:
      n = sum(1 for m in base_meta_rag if m["file"] == doc)
      print(f"  - {doc}: {n} fragmentos")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Construyendo índice RAG para los documentos base...
Índice RAG listo: 61 fragmentos
  - attention_is_all_you_need.pdf: 61 fragmentos


In [11]:

# ============================================================
# VERSIÓN 4 — Asistente RAG: Mejoras A + B + C + D integradas
# ============================================================

def rebuild_index_rag(pdf_file, use_defaults: bool) -> str:
    """Reconstruye el índice RAG con los documentos activos."""
    global current_index_rag
    docs = {}
    if use_defaults:
        docs.update(base_docs)
    if pdf_file is not None:
        nombre = os.path.basename(pdf_file)
        try:
            docs[nombre] = extract_text_from_pdf(pdf_file)
        except Exception as e:
            return f"Error al leer el PDF: {e}"
    if not docs:
        current_index_rag = ([], None, [])
        return "Índice vacío. Sube al menos un PDF o activa los documentos base."
    chunks, embeddings, meta = build_index(docs)
    current_index_rag = (chunks, embeddings, meta)
    nombres = ", ".join(docs.keys())
    return (
        f"**Índice RAG actualizado:** `{len(chunks)}` fragmentos — "
        f"documentos: {nombres}"
    )


def chat_rag_v4(message: str, history: list, pdf_file, use_defaults: bool):
    """Chat RAG: recupera fragmentos relevantes y responde con citas (Mejoras A+B+C+D)."""
    chunks, embeddings, meta = current_index_rag

    # Recuperar los k fragmentos más relevantes para la pregunta
    resultados = buscar_chunks(message, chunks, embeddings, meta, k=4)

    if not resultados:
        yield "No hay documentos indexados. Sube un PDF o activa los documentos base.", \
              "**Sin documentos indexados.**"
        return

    # Construir contexto con etiquetas de fragmento y fuente (Mejoras C + D)
    context_parts = []
    for i, (texto, m, sim) in enumerate(resultados, 1):
        context_parts.append(
            f"[FRAGMENTO {i} — {m['file']} — relevancia {sim:.2f}]\n{texto}"
        )
    context = "\n\n".join(context_parts)

    # Conteo real de tokens del contexto recuperado (Mejora A)
    tokens_ctx = client.models.count_tokens(model=MODELO, contents=context).total_tokens
    token_info = (
        f"**Tokens del contexto RAG:** `{tokens_ctx:,}` / `{LIMITE:,}` "
        f"({tokens_ctx / LIMITE * 100:.2f}%) — "
        f"`{len(resultados)}` fragmentos recuperados de `{len(chunks)}` totales"
    )

    # System prompt con instrucción de citar fragmentos (Mejora C)
    system_prompt = f"""Eres un asistente experto en análisis de documentos académicos.

A continuación se te proporcionan FRAGMENTOS RELEVANTES recuperados de los documentos indexados.
Estos fragmentos fueron seleccionados por similitud semántica con la pregunta del usuario.

{context}

INSTRUCCIONES IMPORTANTES:
1. Responde EXCLUSIVAMENTE con información de los fragmentos proporcionados arriba.
2. SIEMPRE cita el fragmento fuente con el formato [FRAGMENTO N — nombre_archivo.pdf].
   Ejemplo: "Según [FRAGMENTO 2 — attention_is_all_you_need.pdf], el modelo usa 6 capas."
3. Incluye citas textuales (entre comillas) del fragmento cuando sea posible.
4. Si la respuesta no está en los fragmentos, responde:
   "La información no se encuentra en los fragmentos recuperados."
5. NO uses tu conocimiento general; cíñete a los fragmentos proporcionados.
6. Responde en el mismo idioma que el usuario."""

    contenido = []
    for entry in history:
        if isinstance(entry, dict):
            role = "model" if entry["role"] == "assistant" else "user"
            contenido.append(
                types.Content(role=role, parts=[types.Part(text=get_text(entry["content"]))])
            )
        elif isinstance(entry, (list, tuple)) and len(entry) == 2:
            u, a = entry
            contenido.append(types.Content(role="user", parts=[types.Part(text=get_text(u))]))
            if a:
                contenido.append(types.Content(role="model", parts=[types.Part(text=get_text(a))]))
    contenido.append(types.Content(role="user", parts=[types.Part(text=message)]))

    respuesta = ""
    for chunk_stream in client.models.generate_content_stream(
        model=MODELO,
        config=types.GenerateContentConfig(system_instruction=system_prompt),
        contents=contenido
    ):
        if chunk_stream.text:
            respuesta += chunk_stream.text
            yield respuesta, token_info


# Usar el índice construido en la celda anterior
current_index_rag = (base_chunks_rag, base_embeddings_rag, base_meta_rag)

tok_rag_inicial = (
    f"**Índice RAG base:** `{len(base_chunks_rag)}` fragmentos de "
    f"{', '.join(base_docs.keys()) if base_docs else 'ningún documento'}"
)

with gr.Blocks(title="Asistente RAG V4") as demo_rag:
    gr.Markdown("## Asistente RAG V4 — Mejoras A + B + C + D")
    gr.Markdown(
        "Este asistente usa **RAG real**: divide los documentos en fragmentos, "
        "calcula embeddings semánticos y recupera solo los más relevantes para cada pregunta.\n\n"
        "- **Mejora A**: conteo real de tokens del contexto recuperado\n"
        "- **Mejora B**: sube cualquier PDF y reconstruye el índice automáticamente\n"
        "- **Mejora C**: cita el fragmento fuente `[FRAGMENTO N — archivo.pdf]` en cada respuesta\n"
        "- **Mejora D**: combina múltiples documentos con etiqueta de origen"
    )
    token_display_rag = gr.Markdown(tok_rag_inicial)
    chat_rag = gr.ChatInterface(
        fn=chat_rag_v4,
        additional_inputs=[
            gr.File(label="Agregar PDF adicional (opcional)", file_types=[".pdf"]),
            gr.Checkbox(label="Incluir documentos base de la carpeta data/", value=True),
        ],
        additional_outputs=[token_display_rag],
        examples=[
            ["¿Cuál es la arquitectura principal del transformer?", None, True],
            ["¿Qué es el mecanismo de atención multi-cabeza?", None, True],
            ["¿Cuántas capas tiene el encoder del modelo base?", None, True],
            ["¿Cuáles son los resultados en WMT 2014?", None, True],
        ],
        flagging_mode="never",
    )
    chat_rag.additional_inputs[0].change(
        fn=rebuild_index_rag,
        inputs=[chat_rag.additional_inputs[0], chat_rag.additional_inputs[1]],
        outputs=[token_display_rag],
    )
    chat_rag.additional_inputs[1].change(
        fn=rebuild_index_rag,
        inputs=[chat_rag.additional_inputs[0], chat_rag.additional_inputs[1]],
        outputs=[token_display_rag],
    )

demo_rag.launch(server_name="0.0.0.0", server_port=8083, show_error=True)


* Running on local URL:  http://0.0.0.0:8083
* To create a public link, set `share=True` in `launch()`.
